# Chapter 26 — Clustering

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 26 (`code/ch26/gen_seg.py` in the repository).

In [2]:
import numpy as np, pandas as pd
rng = np.random.default_rng(26)

# Four real customer types, plus a scatter of people who fit none of them.
specs = [("bargain hunters",  900, [ 18,  2.1,  46,  1.4]),
         ("weekly regulars", 1100, [ 34,  8.6,  12,  3.2]),
         ("bulk buyers",      700, [128,  1.4,   9,  6.1]),
         ("lapsed",           600, [ 22,  0.4,  71,  1.1])]
rows, truth = [], []
for name, n, (basket, freq, recency, lines) in specs:
    rows.append(np.c_[
        np.exp(rng.normal(np.log(basket), 0.30, n)),
        np.abs(rng.normal(freq, freq * 0.28, n)),
        np.abs(rng.normal(recency, recency * 0.30, n)),
        np.abs(rng.normal(lines, lines * 0.25, n))])
    truth += [name] * n
noise = 200
rows.append(np.c_[np.exp(rng.normal(np.log(45), 1.0, noise)),
                  np.abs(rng.normal(4, 3, noise)),
                  np.abs(rng.normal(40, 30, noise)),
                  np.abs(rng.normal(3, 2, noise))])
truth += ["unclassifiable"] * noise

X = np.vstack(rows)
df = pd.DataFrame(X, columns=["AvgBasket", "OrdersPerMonth",
                              "DaysSinceLast", "LinesPerOrder"]).round(2)
df["TrueType"] = truth
df = df.sample(frac=1, random_state=0).reset_index(drop=True)
df.to_csv("segments.csv", index=False)
print(f"wrote segments.csv: {len(df):,} customers, "
      f"{df.TrueType.nunique()} true groups "
      f"({(df.TrueType == 'unclassifiable').sum()} belong to none)")

wrote segments.csv: 3,500 customers, 5 true groups (200 belong to none)


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch26/_lib.py`.

In [3]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, adjusted_rand_score,
                             calinski_harabasz_score)
# segments.csv is created by this chapter's Step 1, gen_seg.py. The blocks read it from the
# working directory exactly as the book does; if it is not here yet, use the copy shipped in
# data/generated/ (byte-identical to what the generator writes).
import os as _os, shutil as _shutil
if not _os.path.exists("segments.csv"):
    for _d in ("../../data/generated", "../data/generated", "data/generated"):
        if _os.path.exists(_os.path.join(_d, "segments.csv")):
            _shutil.copy(_os.path.join(_d, "segments.csv"), "segments.csv"); break
df = pd.read_csv("segments.csv")
truth = df.pop("TrueType").values
FEATS = list(df.columns)
Xraw = df.values
X = StandardScaler().fit_transform(Xraw)

## The chapter code

### Block 1  (`c1.py`)

In [4]:
# Scaling is not optional here. k-means minimizes squared distance, so a
# column measured in larger numbers dominates the geometry.
print(f"{'feature':<16}{'mean':>10}{'std':>10}{'range':>12}")
for i, f in enumerate(FEATS):
    c = Xraw[:, i]
    print(f"{f:<16}{c.mean():>10.1f}{c.std():>10.1f}"
          f"{c.max() - c.min():>12.1f}")

for label, data in [("unscaled", Xraw), ("standardized", X)]:
    km = KMeans(4, n_init=10, random_state=0).fit(data)
    print(f"\n{label:<13} ARI against the true types: "
          f"{adjusted_rand_score(truth, km.labels_):.3f}")

feature               mean       std       range
AvgBasket             50.5      49.5       514.5
OrdersPerMonth         3.9       3.7        15.3
DaysSinceLast         32.0      26.8       138.3
LinesPerOrder          2.9       2.0        11.2



unscaled      ARI against the true types: 0.533



standardized  ARI against the true types: 0.753


### Block 2  (`c2.py`)

In [5]:
# Choosing k. Inertia always falls, so it cannot pick k on its own.
print(f"{'k':>3}{'inertia':>12}{'silhouette':>13}"
      f"{'Calinski-H':>13}{'ARI':>8}")
for k in range(2, 9):
    km = KMeans(k, n_init=10, random_state=0).fit(X)
    print(f"{k:>3}{km.inertia_:>12,.0f}"
          f"{silhouette_score(X, km.labels_):>13.3f}"
          f"{calinski_harabasz_score(X, km.labels_):>13.0f}"
          f"{adjusted_rand_score(truth, km.labels_):>8.3f}")
print("\nthe true answer is 4 clusters plus 200 customers in none of them")

  k     inertia   silhouette   Calinski-H     ARI


  2       7,658        0.464         2897   0.459


  3       3,424        0.585         5400   0.694


  4       2,721        0.498         4830   0.753


  5       2,359        0.455         4312   0.706


  6       2,040        0.381         4097   0.575


  7       1,835        0.376         3860   0.544


  8       1,664        0.324         3697   0.471

the true answer is 4 clusters plus 200 customers in none of them


### Block 3  (`c3.py`)

In [6]:
# Three algorithms, same data, same scaling.
km = KMeans(4, n_init=10, random_state=0).fit(X)
hc = AgglomerativeClustering(4, linkage="ward").fit(X)
db = DBSCAN(eps=0.55, min_samples=12).fit(X)

for name, lab in [("k-means (k=4)", km.labels_),
                  ("hierarchical (ward)", hc.labels_),
                  ("DBSCAN", db.labels_)]:
    n_found = len(set(lab)) - (1 if -1 in lab else 0)
    noise = int((lab == -1).sum())
    sil = silhouette_score(X[lab != -1],
                           lab[lab != -1]) if n_found > 1 else 0
    print(f"{name:<22} clusters {n_found}   noise {noise:>4}   "
          f"ARI {adjusted_rand_score(truth, lab):.3f}   sil {sil:.3f}")

# Only DBSCAN can decline to assign a point. How good is it at that?
flagged = db.labels_ == -1
real_noise = truth == "unclassifiable"
print(f"\nof {flagged.sum()} points DBSCAN called noise, "
      f"{int((flagged & real_noise).sum())} were genuinely unclassifiable")
print(f"of {real_noise.sum()} genuinely unclassifiable customers, "
      f"DBSCAN caught {int((flagged & real_noise).sum())}")

k-means (k=4)          clusters 4   noise    0   ARI 0.753   sil 0.498
hierarchical (ward)    clusters 4   noise    0   ARI 0.805   sil 0.487
DBSCAN                 clusters 1   noise  130   ARI 0.038   sil 0.000

of 130 points DBSCAN called noise, 116 were genuinely unclassifiable
of 200 genuinely unclassifiable customers, DBSCAN caught 116


### Block 4  (`c4.py`)

In [7]:
# DBSCAN has no k, but it has eps -- and it is far more sensitive to eps
# than k-means is to k.
real_noise = truth == "unclassifiable"
print(f"{'eps':>6}{'clusters':>10}{'noise':>8}{'ARI':>8}"
      f"{'noise precision':>17}")
for eps in (0.25, 0.30, 0.35, 0.40, 0.45, 0.55):
    db = DBSCAN(eps=eps, min_samples=12).fit(X)
    lab = db.labels_
    n = len(set(lab)) - (1 if -1 in lab else 0)
    flagged = lab == -1
    prec = (flagged & real_noise).sum() / max(flagged.sum(), 1)
    print(f"{eps:>6.2f}{n:>10}{int(flagged.sum()):>8}"
          f"{adjusted_rand_score(truth, lab):>8.3f}{prec:>17.1%}")

   eps  clusters   noise     ARI  noise precision
  0.25         4     728   0.549            25.7%
  0.30         4     435   0.656            39.5%
  0.35         4     308   0.704            53.6%


  0.40         3     228   0.728            67.1%
  0.45         3     174   0.306            79.9%
  0.55         1     130   0.038            89.2%


### Block 5  (`c5.py`)

In [8]:
# k-means assumes clusters are round blobs of similar size. When they are
# not, it fails in a way no choice of k repairs.
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=1200, noise=0.06, random_state=0)
Xm = StandardScaler().fit_transform(Xm)

def ward(k):
    return AgglomerativeClustering(k, linkage="ward").fit_predict(Xm)
def single(k):
    return AgglomerativeClustering(k, linkage="single").fit_predict(Xm)

runs = [("k-means (k=2)",
         KMeans(2, n_init=10, random_state=0).fit_predict(Xm)),
        ("hierarchical (ward)", ward(2)),
        ("hierarchical (single)", single(2)),
        ("DBSCAN", DBSCAN(eps=0.30, min_samples=8).fit_predict(Xm))]

for name, lab in runs:
    n = len(set(lab)) - (1 if -1 in lab else 0)
    print(f"{name:<24}clusters {n}   "
          f"ARI {adjusted_rand_score(ym, lab):.3f}")
print("\nthe two moons are the same size and equally dense --")
print("only the SHAPE defeats k-means.")

k-means (k=2)           clusters 2   ARI 0.471
hierarchical (ward)     clusters 2   ARI 0.497
hierarchical (single)   clusters 2   ARI 1.000
DBSCAN                  clusters 2   ARI 1.000

the two moons are the same size and equally dense --
only the SHAPE defeats k-means.


### Block 6  (`c6.py`)

In [9]:
# The deliverable is not the labels. It is the profile.
km = KMeans(4, n_init=10, random_state=0).fit(X)
prof = df.copy()
prof["cluster"] = km.labels_
overall = prof[FEATS].mean()

print(f"{'cluster':>8}{'n':>7}" + "".join(f"{f:>16}" for f in FEATS))
for c in range(4):
    sub = prof[prof.cluster == c]
    cells = "".join(f"{sub[f].mean():>16.1f}" for f in FEATS)
    print(f"{c:>8}{len(sub):>7}{cells}")
print(f"{'ALL':>8}{len(prof):>7}" +
      "".join(f"{overall[f]:>16.1f}" for f in FEATS))

print(f"\nindex against the average (100 = typical customer)")
print(f"{'cluster':>8}" + "".join(f"{f:>16}" for f in FEATS))
for c in range(4):
    sub = prof[prof.cluster == c]
    cells = "".join(f"{100*sub[f].mean()/overall[f]:>16.0f}" for f in FEATS)
    print(f"{c:>8}{cells}")

 cluster      n       AvgBasket  OrdersPerMonth   DaysSinceLast   LinesPerOrder
       0   1049            21.0             1.9            42.2             1.4
       1    744           133.7             1.6            11.0             6.0
       2   1142            36.3             8.6            12.9             3.2
       3    565            24.0             0.9            79.0             1.2
     ALL   3500            50.5             3.9            32.0             2.9

index against the average (100 = typical customer)
 cluster       AvgBasket  OrdersPerMonth   DaysSinceLast   LinesPerOrder
       0              42              49             132              48
       1             265              41              34             203
       2              72             224              40             109
       3              48              23             247              41
